# PDA Platform Demo Pipeline

Smoke-test the running FastAPI backend from a notebook. Start the backend first, then run these cells.

In [ ]:
import datetime as dt
import json
from pathlib import Path

import requests

BASE_URL = "http://localhost:8000"
AOI = {
    "type": "Polygon",
    "coordinates": [[[-115.5, 35.9], [-115.1, 35.9], [-115.1, 36.3], [-115.5, 36.3], [-115.5, 35.9]]],
}

def show_response(response):
    print(response.status_code)
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:1000])

In [ ]:
response = requests.get(f"{BASE_URL}/health", timeout=10)
show_response(response)

## Ingest Social Posts

These use the backend sentiment pipeline. If transformer models are not installed locally, the backend falls back to the deterministic lexicon analyzer.

In [ ]:
tweets = [
    {
        "tweet_id": "demo-001",
        "text": "City center is flooded and people are trapped near the bridge. We need help urgently.",
        "user_location": "Las Vegas, NV",
        "timestamp": dt.datetime.utcnow().isoformat(),
        "language": "en",
    },
    {
        "tweet_id": "demo-002",
        "text": "Volunteers reopened the shelter and families are safe tonight.",
        "user_location": "Clark County, NV",
        "timestamp": dt.datetime.utcnow().isoformat(),
        "language": "en",
    },
]

response = requests.post(f"{BASE_URL}/ingest/tweets/batch", json=tweets, timeout=30)
show_response(response)

## Query RAG Commentary

The RAG endpoint now works even before documents are indexed by using built-in disaster-response baseline chunks.

In [ ]:
payload = {
    "query": "What are flood response priorities for a major damage zone?",
    "damage_context": {"severity": "major", "affected_area": "sector_7"},
    "top_k": 5,
}
response = requests.post(f"{BASE_URL}/rag/query", json=payload, timeout=30)
show_response(response)

In [ ]:
response = requests.get(f"{BASE_URL}/dashboard/metrics", timeout=30)
show_response(response)